🥇 **Camada Gold**

Criação do banco de dados:

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS gold;

Criação da tabela gold.fat_vendas_comercial: 

In [0]:
%sql

CREATE OR REPLACE TABLE gold.fat_vendas_comercial AS

SELECT
    YEAR(p.data_pedido) AS ano_venda,
    MONTH(p.data_pedido) AS mes_venda,
    COALESCE(pr.categoria_produto, 'SEM CATEGORIA') AS categoria_produto,

    COUNT(DISTINCT p.id_pedido) AS total_pedidos,

    COUNT(i.id_item) AS qtd_itens_vendidos,

    ROUND(SUM(i.preco_BRL), 2) AS receita_total_brl,

    ROUND(SUM(
        (i.preco_BRL / NULLIF(p.valor_total_pago_brl, 0)) 
        * COALESCE(p.valor_total_pago_usd, 0)
    ), 2) AS receita_total_usd,

    ROUND(
        SUM(i.preco_BRL) / COUNT(DISTINCT p.id_pedido),
        2
    ) AS ticket_medio_brl

FROM silver.fat_pedido_total p

JOIN silver.fat_itens_pedidos i
    ON p.id_pedido = i.id_pedido

JOIN silver.dim_produtos pr
    ON i.id_produto = pr.id_produto

GROUP BY
    YEAR(p.data_pedido),
    MONTH(p.data_pedido),
    pr.categoria_produto

num_affected_rows,num_inserted_rows


Exibe tabela gold.fat_vendas_comercial:

In [0]:
display(spark.table("gold.fat_vendas_comercial"))

ano_venda,mes_venda,categoria_produto,total_pedidos,qtd_itens_vendidos,receita_total_brl,receita_total_usd,ticket_medio_brl
2017,6,informatica_acessorios,219,261,37007.08,11250.82,168.98
2018,3,cama_mesa_banho,670,798,69256.39,21105.28,103.37
2017,3,cama_mesa_banho,249,289,25773.02,8239.87,103.51
2017,8,informatica_acessorios,297,350,35025.72,11118.97,117.93
2017,7,eletronicos,87,97,7346.84,2291.99,84.45
2017,12,informatica_acessorios,252,287,37880.65,11508.16,150.32
2017,5,eletronicos,62,68,6709.11,2100.23,108.21
2018,7,alimentos,59,64,3378.17,887.31,57.26
2018,5,cine_foto,17,18,2277.29,630.54,133.96
2017,1,automotivo,30,34,5218.53,1642.78,173.95


Exibe os cinco produtos mais vendidos: 

In [0]:
df_top_5_mais_vendidos = spark.sql("""
    SELECT
        COALESCE(p.nome_produto, 'SEM NOME') AS nome_produto,
        COALESCE(p.categoria_produto, 'SEM CATEGORIA') AS categoria_produto,
        COUNT(*) AS quantidade_vendida
    FROM silver.fat_itens_pedidos i
    LEFT JOIN silver.dim_produtos p
        ON i.id_produto = p.id_produto
    GROUP BY
        COALESCE(p.nome_produto, 'SEM NOME'),
        COALESCE(p.categoria_produto, 'SEM CATEGORIA')
    ORDER BY quantidade_vendida DESC, nome_produto ASC
    LIMIT 5
""")

display(df_top_5_mais_vendidos)

nome_produto,categoria_produto,quantidade_vendida
Secador de Cabelo,beleza_saude,1076
Toalha de Banho,cama_mesa_banho,919
Protetor Solar,beleza_saude,917
Travesseiro,cama_mesa_banho,839
Colar,relogios_presentes,804


Exibe os cinco produtos menos vendidos: 

In [0]:
df_top_5_menos_vendidos = spark.sql("""
    SELECT
        COALESCE(p.nome_produto, 'SEM NOME') AS nome_produto,
        COALESCE(p.categoria_produto, 'SEM CATEGORIA') AS categoria_produto,
        COUNT(*) AS quantidade_vendida
    FROM silver.fat_itens_pedidos i
    LEFT JOIN silver.dim_produtos p
        ON i.id_produto = p.id_produto
    GROUP BY
        COALESCE(p.nome_produto, 'SEM NOME'),
        COALESCE(p.categoria_produto, 'SEM CATEGORIA')
    ORDER BY quantidade_vendida ASC, nome_produto ASC
    LIMIT 5
""")

display(df_top_5_menos_vendidos)

nome_produto,categoria_produto,quantidade_vendida
"""Smart TV 50"""" Básico""",eletronicos,1
"""Smart TV 50"""" Master""",eletronicos,1
Acessório Padrão,artigos_de_festas,1
Acessório Padrão,livros_importados,1
Acessório Padrão,fashion_roupa_infanto_juvenil,1


Criação da tabela gold.fat_avaliacoes_clientes:

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fat_avaliacoes_clientes AS
SELECT
    COALESCE(p.categoria_produto, 'SEM CATEGORIA') AS categoria_produto,
    COALESCE(v.nome_vendedor, 'SEM NOME') AS nome_vendedor,
    COALESCE(v.estado, 'SEM ESTADO') AS estado_vendedor,

    COUNT(*) AS total_avaliacoes,

    ROUND(AVG(a.nota_avaliacao), 2) AS avaliacao_media,

    SUM(CASE WHEN a.nota_avaliacao >= 4 THEN 1 ELSE 0 END) AS total_avaliacoes_positivas,

    SUM(CASE WHEN a.nota_avaliacao <= 2 THEN 1 ELSE 0 END) AS total_avaliacoes_negativas,

    ROUND(
        100.0 * SUM(CASE WHEN a.nota_avaliacao >= 4 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS percentual_satisfacao

FROM silver.fat_avaliacoes_pedidos a
JOIN silver.fat_itens_pedidos i
    ON a.id_pedido = i.id_pedido
LEFT JOIN silver.dim_produtos p
    ON i.id_produto = p.id_produto
LEFT JOIN silver.dim_vendedores v
    ON i.id_vendedor = v.id_vendedor

GROUP BY
    COALESCE(p.categoria_produto, 'SEM CATEGORIA'),
    COALESCE(v.nome_vendedor, 'SEM NOME'),
    COALESCE(v.estado, 'SEM ESTADO');

num_affected_rows,num_inserted_rows


Exibe tabela gold.fat_avaliacoes_clientes:

In [0]:
display(spark.table("gold.fat_avaliacoes_clientes"))

categoria_produto,nome_vendedor,estado_vendedor,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
esporte_lazer,Emanuelly Mendonça,SP,399,4.15,317,52,79.45
ferramentas_jardim,Maya Cirino,SP,7,4.43,7,0,100.00
cool_stuff,Milena Moreira,RS,125,4.54,114,6,91.20
pet_shop,Bárbara Pacheco,RJ,97,4.2,77,18,79.38
brinquedos,Isabela Mendes,RJ,355,4.23,291,42,81.97
informatica_acessorios,Rael Lima,MG,326,4.22,268,37,82.21
cama_mesa_banho,Sra. Sara Rezende,SP,132,4.31,112,18,84.85
automotivo,Gabriel Aparecida,SP,11,4.55,10,1,90.91
esporte_lazer,Raul Peixoto,SP,297,4.16,234,40,78.79
fashion_bolsas_e_acessorios,Hellena Castro,DF,33,3.91,25,6,75.76


Exibe o produto mais bem avaliado:

In [0]:
df_produto_mais_bem_avaliado = spark.sql("""
    WITH base AS (
        SELECT
            i.id_produto,
            COALESCE(p.nome_produto, 'SEM NOME') AS nome_produto,
            COALESCE(p.categoria_produto, 'SEM CATEGORIA') AS categoria_produto,
            a.nota_avaliacao
        FROM silver.fat_avaliacoes_pedidos a
        JOIN silver.fat_itens_pedidos i
            ON a.id_pedido = i.id_pedido
        LEFT JOIN silver.dim_produtos p
            ON i.id_produto = p.id_produto
    )
    SELECT
        id_produto,
        nome_produto,
        categoria_produto,
        COUNT(*) AS total_avaliacoes,
        ROUND(AVG(nota_avaliacao), 2) AS avaliacao_media
    FROM base
    GROUP BY
        id_produto,
        nome_produto,
        categoria_produto
    ORDER BY
        avaliacao_media DESC,
        total_avaliacoes DESC
    LIMIT 1
""")

display(df_produto_mais_bem_avaliado)

id_produto,nome_produto,categoria_produto,total_avaliacoes,avaliacao_media
37eb69aca8718e843d897aa7b82f462d,Kit de Ferramentas Dourado,ferramentas_jardim,15,5.0


Exibe produto menos bem avaliado: 

In [0]:
df_produto_menos_bem_avaliado = spark.sql("""
    WITH base AS (
        SELECT
            i.id_produto,
            COALESCE(p.nome_produto, 'SEM NOME') AS nome_produto,
            COALESCE(p.categoria_produto, 'SEM CATEGORIA') AS categoria_produto,
            a.nota_avaliacao
        FROM silver.fat_avaliacoes_pedidos a
        JOIN silver.fat_itens_pedidos i
            ON a.id_pedido = i.id_pedido
        LEFT JOIN silver.dim_produtos p
            ON i.id_produto = p.id_produto
    )
    SELECT
        id_produto,
        nome_produto,
        categoria_produto,
        COUNT(*) AS total_avaliacoes,
        ROUND(AVG(nota_avaliacao), 2) AS avaliacao_media
    FROM base
    GROUP BY
        id_produto,
        nome_produto,
        categoria_produto
    ORDER BY
        avaliacao_media ASC,
        total_avaliacoes DESC
    LIMIT 1
""")

display(df_produto_menos_bem_avaliado)

id_produto,nome_produto,categoria_produto,total_avaliacoes,avaliacao_media
270516a3f41dc035aa87d220228f844c,Protetor Solar,beleza_saude,10,1.0


Exibe o vendedor mais bem avaliado: 

In [0]:
df_vendedor_mais_bem_avaliado = spark.sql("""
    WITH base AS (
        SELECT
            i.id_vendedor,
            COALESCE(v.nome_vendedor, 'SEM NOME') AS nome_vendedor,
            COALESCE(v.estado, 'SEM ESTADO') AS estado_vendedor,
            a.nota_avaliacao
        FROM silver.fat_avaliacoes_pedidos a
        JOIN silver.fat_itens_pedidos i
            ON a.id_pedido = i.id_pedido
        LEFT JOIN silver.dim_vendedores v
            ON i.id_vendedor = v.id_vendedor
    )
    SELECT
        id_vendedor,
        nome_vendedor,
        estado_vendedor,
        COUNT(*) AS total_avaliacoes,
        ROUND(AVG(nota_avaliacao), 2) AS avaliacao_media
    FROM base
    GROUP BY
        id_vendedor,
        nome_vendedor,
        estado_vendedor
    ORDER BY
        avaliacao_media DESC,
        total_avaliacoes DESC
    LIMIT 1
""")

display(df_vendedor_mais_bem_avaliado)

id_vendedor,nome_vendedor,estado_vendedor,total_avaliacoes,avaliacao_media
48efc9d94a9834137efd9ea76b065a38,Luiz Otávio Abreu,PR,32,5.0


Exibe vendedor menos bem avaliado:

In [0]:
df_vendedor_menos_bem_avaliado = spark.sql("""
    WITH base AS (
        SELECT
            i.id_vendedor,
            COALESCE(v.nome_vendedor, 'SEM NOME') AS nome_vendedor,
            COALESCE(v.estado, 'SEM ESTADO') AS estado_vendedor,
            a.nota_avaliacao
        FROM silver.fat_avaliacoes_pedidos a
        JOIN silver.fat_itens_pedidos i
            ON a.id_pedido = i.id_pedido
        LEFT JOIN silver.dim_vendedores v
            ON i.id_vendedor = v.id_vendedor
    )
    SELECT
        id_vendedor,
        nome_vendedor,
        estado_vendedor,
        COUNT(*) AS total_avaliacoes,
        ROUND(AVG(nota_avaliacao), 2) AS avaliacao_media
    FROM base
    GROUP BY
        id_vendedor,
        nome_vendedor,
        estado_vendedor
    ORDER BY
        avaliacao_media ASC,
        total_avaliacoes DESC
    LIMIT 1
""")

display(df_vendedor_menos_bem_avaliado)

id_vendedor,nome_vendedor,estado_vendedor,total_avaliacoes,avaliacao_media
8d92f3ea807b89465643c219455e7369,Sra. Fernanda Santos,SP,8,1.0
